In [ ]:
# ============================================================
# NOTEBOOK 04
# 3D RESUNET MODEL WITH CLASSIFICATION BRANCH
# ============================================================

import os
import numpy as np
import tensorflow as tf

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    Conv3D,
    BatchNormalization,
    Activation,
    Add,
    MaxPooling3D,
    UpSampling3D,
    Concatenate,
    GlobalAveragePooling3D,
    Dense,
    Dropout
)

In [ ]:
# ============================================================
# CHECK TENSORFLOW AND GPU
# ============================================================

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU available:")
    for gpu in gpus:
        print(gpu)
else:
    print("WARNING: No GPU detected.")

In [ ]:
# ============================================================
# MODEL CONFIGURATION
# ============================================================

PATCH_SIZE = (64, 64, 64)
CHANNELS = 1

INPUT_SHAPE = (
    PATCH_SIZE[0],
    PATCH_SIZE[1],
    PATCH_SIZE[2],
    CHANNELS
)

NUM_CLASSES = 1

print("Input shape:", INPUT_SHAPE)

In [ ]:
# ============================================================
# RESIDUAL BLOCK
# ============================================================

def residual_block(x, filters, name):
    """
    3D residual convolutional block.

    Main branch:
        Conv3D -> BN -> ReLU
        Conv3D -> BN

    Shortcut:
        Identity if channel dimensions match
        1x1x1 Conv3D otherwise

    Output:
        Add(main branch, shortcut) -> ReLU
    """

    shortcut = x

    # First convolution
    x = Conv3D(
        filters=filters,
        kernel_size=(3, 3, 3),
        padding="same",
        name=f"{name}_conv1"
    )(x)

    x = BatchNormalization(
        name=f"{name}_bn1"
    )(x)

    x = Activation(
        "relu",
        name=f"{name}_relu1"
    )(x)

    # Second convolution
    x = Conv3D(
        filters=filters,
        kernel_size=(3, 3, 3),
        padding="same",
        name=f"{name}_conv2"
    )(x)

    x = BatchNormalization(
        name=f"{name}_bn2"
    )(x)

    # Adjust shortcut channels if necessary
    if shortcut.shape[-1] != filters:

        shortcut = Conv3D(
            filters=filters,
            kernel_size=(1, 1, 1),
            padding="same",
            name=f"{name}_shortcut"
        )(shortcut)

        shortcut = BatchNormalization(
            name=f"{name}_shortcut_bn"
        )(shortcut)

    # Residual addition
    x = Add(
        name=f"{name}_add"
    )([x, shortcut])

    x = Activation(
        "relu",
        name=f"{name}_output"
    )(x)

    return x

In [ ]:
# ============================================================
# ENCODER BLOCK
# ============================================================

def encoder_block(x, filters, name):
    """
    ResUNet encoder block.

    Returns:
        feature map for skip connection
        downsampled feature map
    """

    x = residual_block(
        x,
        filters,
        name=f"{name}_res"
    )

    skip = x

    x = MaxPooling3D(
        pool_size=(2, 2, 2),
        name=f"{name}_pool"
    )(x)

    return skip, x

In [ ]:
# ============================================================
# DECODER BLOCK
# ============================================================

def decoder_block(x, skip, filters, name):
    """
    ResUNet decoder block.

    Upsampling
        ↓
    Concatenate with encoder skip connection
        ↓
    Residual block
    """

    x = UpSampling3D(
        size=(2, 2, 2),
        name=f"{name}_upsample"
    )(x)

    x = Concatenate(
        name=f"{name}_concat"
    )([x, skip])

    x = residual_block(
        x,
        filters,
        name=f"{name}_res"
    )

    return x

In [ ]:
# ============================================================
# CLASSIFICATION BRANCH
# ============================================================

def classification_branch(
    bottleneck,
    dropout_rate=0.3
):
    """
    Classification branch for fracture/no-fracture prediction.

    Input:
        3D bottleneck feature representation

    Output:
        Probability of fracture presence
    """

    x = GlobalAveragePooling3D(
        name="classification_global_pool"
    )(bottleneck)

    x = Dense(
        128,
        activation="relu",
        name="classification_dense1"
    )(x)

    x = Dropout(
        dropout_rate,
        name="classification_dropout"
    )(x)

    output = Dense(
        1,
        activation="sigmoid",
        name="classification_output"
    )(x)

    return output

In [ ]:
# ============================================================
# BUILD 3D RESUNET
# ============================================================

def build_3d_resunet(
    input_shape=(64, 64, 64, 1)
):

    inputs = Input(
        shape=input_shape,
        name="ct_input"
    )

    # ========================================================
    # ENCODER
    # ========================================================

    skip1, x = encoder_block(
        inputs,
        16,
        "encoder1"
    )

    skip2, x = encoder_block(
        x,
        32,
        "encoder2"
    )

    skip3, x = encoder_block(
        x,
        64,
        "encoder3"
    )

    # ========================================================
    # BOTTLENECK
    # ========================================================

    bottleneck = residual_block(
        x,
        128,
        "bottleneck"
    )

    # ========================================================
    # CLASSIFICATION BRANCH
    # ========================================================

    classification_output = classification_branch(
        bottleneck
    )

    # ========================================================
    # DECODER
    # ========================================================

    x = decoder_block(
        bottleneck,
        skip3,
        64,
        "decoder3"
    )

    x = decoder_block(
        x,
        skip2,
        32,
        "decoder2"
    )

    x = decoder_block(
        x,
        skip1,
        16,
        "decoder1"
    )

    # ========================================================
    # SEGMENTATION OUTPUT
    # ========================================================

    segmentation_output = Conv3D(
        filters=1,
        kernel_size=(1, 1, 1),
        activation="sigmoid",
        padding="same",
        name="segmentation_output"
    )(x)

    # ========================================================
    # FINAL MODEL
    # ========================================================

    model = Model(
        inputs=inputs,
        outputs=[
            segmentation_output,
            classification_output
        ],
        name="3D_ResUNet_MultiTask"
    )

    return model

In [ ]:
# ============================================================
# CREATE MODEL
# ============================================================

model = build_3d_resunet(
    input_shape=INPUT_SHAPE
)

print("Model created successfully.")

In [ ]:
# ============================================================
# MODEL SUMMARY
# ============================================================

model.summary()

In [ ]:
# ============================================================
# PARAMETER COUNT
# ============================================================

trainable_params = np.sum(
    [
        np.prod(v.shape)
        for v in model.trainable_weights
    ]
)

print(
    "Trainable parameters:",
    trainable_params
)

In [ ]:
# ============================================================
# MODEL FORWARD-PASS TEST
# ============================================================

dummy_input = np.random.rand(
    1,
    64,
    64,
    64,
    1
).astype(np.float32)

seg_output, cls_output = model(
    dummy_input,
    training=False
)

print("Segmentation output shape:")
print(seg_output.shape)

print()

print("Classification output shape:")
print(cls_output.shape)

In [ ]:
# ============================================================
# CHECK OUTPUT RANGES
# ============================================================

print(
    "Segmentation minimum:",
    tf.reduce_min(seg_output).numpy()
)

print(
    "Segmentation maximum:",
    tf.reduce_max(seg_output).numpy()
)

print(
    "Classification prediction:",
    cls_output.numpy()
)

In [ ]:
# ============================================================
# LOAD PATCH DATASET
# ============================================================

PATCH_DATASET_ROOT = "/kaggle/input/YOUR-PATCH-DATASET"

X_PATH = os.path.join(
    PATCH_DATASET_ROOT,
    "X_patches.npy"
)

Y_SEG_PATH = os.path.join(
    PATCH_DATASET_ROOT,
    "Y_segmentation.npy"
)

Y_CLASS_PATH = os.path.join(
    PATCH_DATASET_ROOT,
    "Y_classification.npy"
)

In [ ]:
# ============================================================
# LOAD PATCHES
# ============================================================

X = np.load(
    X_PATH
)

Y_seg = np.load(
    Y_SEG_PATH
)

Y_class = np.load(
    Y_CLASS_PATH
)

print("X:", X.shape)
print("Y_seg:", Y_seg.shape)
print("Y_class:", Y_class.shape)

In [ ]:
# ============================================================
# DATA-MODEL COMPATIBILITY
# ============================================================

assert X.shape[1:] == INPUT_SHAPE

assert Y_seg.shape[1:] == (
    64, 64, 64, 1
)

assert Y_class.ndim == 1

print("Data and model dimensions are compatible.")

In [ ]:
# ============================================================
# COMPILE MODEL
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss={
        "segmentation_output": "binary_crossentropy",
        "classification_output": "binary_crossentropy"
    },

    loss_weights={
        "segmentation_output": 1.0,
        "classification_output": 1.0
    },

    metrics={
        "segmentation_output": [
            "accuracy"
        ],

        "classification_output": [
            "accuracy",
            tf.keras.metrics.AUC(
                name="auc"
            )
        ]
    }
)

print("Model compiled successfully.")

In [ ]:
# ============================================================
# CHECK COMPILED MODEL
# ============================================================

print("Model inputs:")
print(model.input)

print()

print("Model outputs:")
print(model.output)

print()

print("Model is ready for training.")

In [ ]:
# ============================================================
# SAVE MODEL ARCHITECTURE
# ============================================================

MODEL_DIR = "/kaggle/working/model"

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

MODEL_JSON_PATH = os.path.join(
    MODEL_DIR,
    "3D_ResUNet_MultiTask.json"
)

with open(
    MODEL_JSON_PATH,
    "w"
) as json_file:

    json_file.write(
        model.to_json()
    )

print("Model architecture saved:")
print(MODEL_JSON_PATH)

In [ ]:
# ============================================================
# FINAL NOTEBOOK 04 VALIDATION
# ============================================================

assert model.input_shape == (
    None,
    64,
    64,
    64,
    1
)

assert len(model.outputs) == 2

assert model.output_shape[0] == (
    None,
    64,
    64,
    64,
    1
)

assert model.output_shape[1] == (
    None,
    1
)

print("==============================================")
print("NOTEBOOK 04 - MODEL CONSTRUCTION SUCCESSFUL")
print("==============================================")

print("Model:", model.name)

print("Input:", model.input_shape)

print("Segmentation output:")
print(model.output_shape[0])

print("Classification output:")
print(model.output_shape[1])

print("==============================================")